# Lecture 01: Extended tokenization exercises

Baojian Zhou · Fudan University · Fall 2026

Explore Ollama, Unicode, regular expressions, spaCy, datasets, BPE, WordPiece, and pretrained tokenizers. Use the separate `lecture-01-exercise.ipynb` for the timed classroom exercises E01–E06.

Adapted from the [Spring 2026 tokenization notebook](https://github.com/baojian/llm-26/blob/main/lecture-01-tokenization/lecture-01-exercise-tokenization.ipynb).

---

<!-- bilingual-zh -->

# 第 1 讲：词元化扩展练习

周保健 · 复旦大学 · 2026 年秋季。探索 Ollama、Unicode、正则表达式、spaCy、数据集、BPE、WordPiece 和预训练词元化器。课堂计时练习 E01–E06 请使用另一份 `lecture-01-exercise.ipynb`。本材料改编自原文链接的 2026 年春季 notebook。


In [ ]:
from IPython.display import display, HTML

display(HTML("""
<style>
/* JupyterLab: center notebook and leave margins */
.jp-NotebookPanel-notebook {
  width: 85% !important;              
  margin-left: auto !important;
  margin-right: auto !important;
  max-width: 1100px !important;       /* optional cap */
}

/* Make output area match nicely */
.jp-OutputArea {
  max-width: 100% !important;
}
</style>
"""))
# make high-resolution of images.
%config InlineBackend.figure_format = 'svg'
print("setup jnk layout and figure format.")

<a id="0"></a> <br>
## Table of Contents
- [0. Preparing Your Environment](#00)
  - [0.1 Set up the course uv environment](#01)
  - [0.2 Checking your device](#02)
- [1. Explore LLMs via ollama](#10)
  - [1.1 Install ollama and download LLMs](#11)
  - [1.2 Get response for a given prompt](#12)
  - [1.3 Get response probability from LLMs](#13)
- [2. Basics for Python and spaCy](#20)
  - [2.1 Unicode for text data](#21)
  - [2.2 Python basics: regular expression](#22)
  - [2.3 (Old way) Tokenization tool: spaCy](#23)
- [3. Datasets Exploration](#30)
  - [3.1 Heap's Law](#31)
  - [3.2 Explore wikitext-2 dataset](#32)
  - [3.3 Explore wikitext-103 dataset](#33)
  - [3.4 BookCorpus datasets](#34)
- [4. LLMs tokenization (Modern Way)](#40)
  - [4.1 Subword tokenization: BPE](#41)
  - [4.2 Subword tokenization: WordPiece](#42)
  - [4.3 Subword tokenization: Unigram](#43)
- [5. Save Your Work](#50)
- [6. Useful Commands and References](#60)

---

<!-- bilingual-zh -->

## 中文目录

- [0. 准备环境](#00)：[uv 环境](#01)、[检查设备](#02)
- [1. 通过 Ollama 探索大语言模型](#10)：[安装与模型](#11)、[提示与响应](#12)、[响应概率](#13)
- [2. Python 与 spaCy 基础](#20)：[Unicode](#21)、[正则表达式](#22)、[spaCy 词元化](#23)
- [3. 数据集探索](#30)：[Heaps 定律](#31)、[WikiText-2](#32)、[WikiText-103](#33)、[BookCorpus](#34)
- [4. 现代大语言模型词元化](#40)：[BPE](#41)、[WordPiece](#42)、[Unigram](#43)
- [5. 保存工作](#50) · [6. 命令与参考](#60)


<a id="00"></a> 
## 0. Preparing Your Environment

---

<!-- bilingual-zh -->

## 0. 准备环境


<a id="01"></a>
### 0.1 Set up the course uv environment

---

<!-- bilingual-zh -->

### 0.1 配置课程 uv 环境


Run this notebook in your personal copy under `workspace/slides/lecture-01/`. Install the optional packages before starting JupyterLab. Ollama, spaCy language models, and remote datasets are prepared separately in the sections below.

---

<!-- bilingual-zh -->

请在 `workspace/slides/lecture-01/` 下的个人副本中运行本 notebook，并在启动 JupyterLab 前安装可选依赖。Ollama、spaCy 语言模型和远程数据集需按下文分别准备。


From the repository root, install the extended notebook dependencies:

```sh
uv sync --extra tokenization
uv run --extra tokenization python scripts/slides.py serve
```

Open the Lecture 01 slides and follow **Extended tokenization notebook**. The launcher creates a personal copy and opens it in the course environment. If JupyterLab was already running, restart its kernel after installing packages.

Work through one section at a time. Sections 1, 3, and 4 include local model calls, dataset downloads, and tokenizer training; they take longer than the offline classroom exercises. Run them when the required models and datasets are available. Keep generated tokenizers, exports, and datasets in `workspace/`.

---

<!-- bilingual-zh -->

在仓库根目录使用上方命令安装扩展 notebook 的依赖并启动预览。打开第 1 讲课件，选择 **Extended tokenization notebook**；启动器会创建个人副本并在课程环境中打开。如果 JupyterLab 已运行，安装依赖后需重启内核。

每次学习一个部分。第 1、3、4 节包含本地模型调用、数据集下载和词元化器训练，耗时长于离线课堂练习；仅在相应模型或数据已可用时运行。生成的词元化器、导出文件和数据集保存在 `workspace/`。


- **Load all necessary packages**

  At this point, your env is ready to use. We import all necessary packages here.

---

<!-- bilingual-zh -->

**加载所需包：**环境准备好后，在此导入所需依赖。


In [ ]:
import re
import os
import time
import math

import shutil
import socket
import platform
import requests
import datetime
import tarfile

import spacy
import torch
import numpy as np

from transformers import AutoTokenizer
from transformers import Qwen2Tokenizer, Qwen2TokenizerFast

import tiktoken
from tokenizers import normalizers
from tokenizers.normalizers import NFD, StripAccents
from tokenizers import decoders

from transformers import AutoTokenizer
from tokenizers import Tokenizer
from tokenizers import normalizers
from tokenizers.models import WordPiece
from tokenizers.normalizers import NFD, Lowercase, StripAccents
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing

from datasets import load_dataset
import matplotlib.pyplot as plt

jnk_start_time = time.time()

<a id="02"></a> 
### 0.2 Checking your device

---

<!-- bilingual-zh -->

### 0.2 检查设备


Check the course environment from the repository root with `uv run --extra tokenization python -c "import torch; print(torch.__version__)"`. The next cells report the available CPU, CUDA, or Apple MPS device; a GPU is optional for tokenization.

---

<!-- bilingual-zh -->

在仓库根目录使用原文命令检查课程环境中的 PyTorch 版本。接下来的单元格报告可用 CPU、CUDA 或 Apple MPS 设备；词元化不强制要求 GPU。


In [ ]:
!python -c "import torch; print('torch', torch.__version__); print('cuda?', torch.cuda.is_available()); print('mps?', getattr(torch.backends,'mps',None) is not None and torch.backends.mps.is_available()); print('cuda device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu'); print('using:', 'cuda' if torch.cuda.is_available() else ('mps' if (getattr(torch.backends,'mps',None) is not None and torch.backends.mps.is_available()) else 'cpu'))"

- The following code provides a more symmetric way to perform the same checks. In our lectures, some datasets are very large, so it’s a good idea to make sure you have enough disk space. You can check your disk, memory, and CPU information using the code below.

---

<!-- bilingual-zh -->

下方代码以更统一的方式执行相同检查。一些课程数据集很大，应确认磁盘空间充足；代码还检查磁盘、内存和 CPU 信息。


In [ ]:
def detect_torch_device(verbose: bool = True) -> str:
    """
    Returns one of: 'cuda', 'mps', 'cpu'
    Priority: CUDA GPU > Apple MPS > CPU
    """
    has_cuda = torch.cuda.is_available()
    has_mps = getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available()

    if has_cuda:
        device = "cuda"
    elif has_mps:
        device = "mps"
    else:
        device = "cpu"

    if verbose:
        print(f"torch: {torch.__version__}")
        print(f"device: {device}")

        if has_cuda:
            print(f"cuda devices: {torch.cuda.device_count()}")
            for i in range(torch.cuda.device_count()):
                print(f"  [{i}] {torch.cuda.get_device_name(i)}")
        elif has_mps:
            print("mps available: True (Apple Metal)")
        else:
            print("cpu only")

    return device

device = detect_torch_device()
# generate 2x3 random matrix to check torch
x = torch.rand(2, 3)
x = x.to(device)
print("device:", x.device)

- **Exercise System Info Cell**

---

<!-- bilingual-zh -->

**练习：系统信息单元格。**


In [ ]:
def bytes_to_gb(x: int) -> float:
    return x / (1024 ** 3)

def system_report(path: str = "."):
    print("=== System Report ===")
    print("OS:", platform.system(), platform.release())
    print("Platform:", platform.platform())
    now = datetime.datetime.now().astimezone()
    print("Local time:", now.strftime("%Y-%m-%d %H:%M:%S %Z%z"))
    print("Hostname  :", socket.gethostname())
    print("User      :", os.getenv("USER") or os.getenv("USERNAME") or "unknown")
    
    print("\n=== OS / Python ===")
    print("OS        :", platform.system(), platform.release())
    print("Version   :", platform.version())
    print("Machine   :", platform.machine())
    print("Processor :", platform.processor() or "unknown")
    print("Python    :", platform.python_version())

    # CPU
    print("\n--- CPU ---")
    print("CPU cores (logical):", os.cpu_count())

    # Memory (best-effort, cross-platform)
    print("\n--- Memory (RAM) ---")
    try:
        import psutil  # you already have this in env
        vm = psutil.virtual_memory()
        print(f"Total: {bytes_to_gb(vm.total):.2f} GB")
        print(f"Available: {bytes_to_gb(vm.available):.2f} GB")
        print(f"Used: {bytes_to_gb(vm.used):.2f} GB ({vm.percent}%)")
    except Exception as e:
        print("psutil not available or failed:", e)

    # Disk
    print("\n--- Disk ---")
    total, used, free = shutil.disk_usage(path)
    print("Path checked:", os.path.abspath(path))
    print(f"Total: {bytes_to_gb(total):.2f} GB")
    print(f"Free:  {bytes_to_gb(free):.2f} GB")
    print(f"Used:  {bytes_to_gb(used):.2f} GB")

    # PyTorch device
    print("\n--- PyTorch Device ---")
    try:
        import torch
        cuda = torch.cuda.is_available()
        mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
        print("torch:", torch.__version__)
        print("CUDA available:", cuda)
        print("MPS available:", mps)
        if cuda:
            print("GPU:", torch.cuda.get_device_name(0))
        device = "cuda" if cuda else ("mps" if mps else "cpu")
        print("Suggested device:", device)
    except Exception as e:
        print("torch not available or failed:", e)

system_report(".")

<a id="10"></a> 
## 1. Explore LLMs via ollama

In this section, we’ll have some fun exploring LLMs with [ollama](https://github.com/ollama/ollama).

---

<!-- bilingual-zh -->

## 1. 通过 Ollama 探索大语言模型

本节使用 Ollama 探索模型调用；这是可选部分。


<a id="11"></a> 
### 1.1  Install ollama and download LLMs

We can download some popular LLMs such as Qwen series.


- **Step 1:** Please download and install ollama via [download](https://ollama.com/download), ```curl -fsSL https://ollama.com/install.sh | sh```.

- **Step 2:** After the installation of ollama, you can open a terminal and download qwen3:0.6b and qwen3:1.7b via:

  ```ollama run qwen3:0.6b```

  It will automatically download the model into your local disk. It takes about 498MB disk space. Similarly, you can run a larger version ```qwen3:1.7b``` and it takes about 1.3GB disk space. Let us test 0.6b model

  ```ollama run qwen3:0.6b```

- **Step 2:** Ask a question, like: ```hi, where is Fudan located at?```, it gives us:

<div style="text-align:center;">
<img src="assets/ollama-terminal.png" alt="ollama terminal" style="width:75%;">
</div>

---

<!-- bilingual-zh -->

### 1.1 安装 Ollama 与准备模型

原文说明安装 Ollama 以及通过 `ollama run` 下载并运行 Qwen 系列模型：`qwen3:0.6b` 约占 498 MB，`qwen3:1.7b` 约占 1.3 GB。模型体积和可用性以实际版本为准。随后可在终端询问复旦大学所在城市；示例截图见英文原文。**这些命令会下载模型；阅读本节无需执行。**


<a id="12"></a> 
### 1.2 Get response for a given prompt

---

<!-- bilingual-zh -->

### 1.2 给定提示词获取响应


- **Call Ollama API**

  Next, we go a little deeper: we will call the Ollama API from Python to generate outputs given prompts. You can check whether Ollama is running by typing [http://127.0.0.1:11434/](http://127.0.0.1:11434/) in Chrome.

---

<!-- bilingual-zh -->

**调用 Ollama API：**接下来通过 Python 调用 Ollama API，根据提示生成输出。可以在浏览器访问 `http://127.0.0.1:11434/` 检查本地 Ollama 是否运行。


In [ ]:
# Make sure bypass proxies for local services (e.g., Ollama on localhost:11434)
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"
os.environ["NO_PROXY"] = "localhost,127.0.0.1"
os.environ["no_proxy"] = "localhost,127.0.0.1"

s = requests.Session()
# IMPORTANT <- ignore ALL proxy env vars
s.trust_env = False  
# make sure ollama is running
print(s.get("http://127.0.0.1:11434/api/version", timeout=3).json())

# !!!! make sure to import after the s.trust_env !!!!

import ollama
from ollama import chat
from ollama import ChatResponse

s = requests.Session()
s.trust_env = False  # <- ignore ALL proxy env vars

response: ChatResponse = chat(model='qwen3:0.6b', messages=[
  {
    'role': 'user',
    'content': 'Fudan University is located in which city? Answer with one word.',
  },
])
print(response['message']['content'])

- **Will the same prompt always produce the same output?**

---

<!-- bilingual-zh -->

**相同提示词总会得到相同输出吗？**


In [ ]:
models = ["qwen3:0.6b", "qwen3:1.7b"]
prompt = "Fudan University is located in which city? Answer with one word."

for model in models:
    print('-' * 50)
    start_time = time.time()
    for _ in range(10):
        resp = ollama.generate(
            model = model,
            prompt = prompt
        )
        print(f"{model} with resp {_ + 1}: {resp['response']}")
    print(f'total runtime of 10 responses of {model} is: {time.time() - start_time:.1f} seconds')

- **Some key observations**:
  - The smaller model, **Qwen3:0.6B**, tends to produce lower-quality answers, while the larger model, **Qwen3:1.7B**, generally produces higher-quality answers.
  - The **responses are somewhat random**: running the same prompt multiple times may yield different outputs. There are ways to make the output deterministic. For example, the code below sets decoding options so that the model always selects the highest-probability token at each step.
    <div style="background:#f5f5f5; border:1px solid #ddd; border-radius:8px; max-width:900px; margin:12px auto;">
        <pre style="margin:0;"><code class="language-python">
            resp = ollama.generate(
            model=model,
            prompt=prompt,
            options={
                "temperature": 0.0,
                "top_p": 1.0,
                "top_k": 0,
                # optional:
                "num_predict": 32,
            },
            )
            print(resp['response'])</code></pre>
    </div>
    
  Let us generate some response that are not one word but a sequence of words.

---

<!-- bilingual-zh -->

**主要观察：**原材料称在其示例中，较小的 Qwen3:0.6B 往往不如较大的 Qwen3:1.7B；这不是对所有任务的普遍保证。响应具有一定随机性，同一提示多次运行可能不同。原代码通过将 temperature 设为 0、top_p 设为 1、top_k 设为 0 等方式选择每步最高概率词元；跨模型文件、软件或硬件仍不应保证完全一致。接下来生成不止一个词的回复。


- **Get a paragraph of response with qwen3:1.7b**

---

<!-- bilingual-zh -->

**使用 Qwen3:1.7B 生成一段响应。**


In [ ]:
model = "qwen3:1.7b"
prompt = "I am an undergraduate student, please explain LLMs in three sentences."
resp = ollama.generate(
            model=model,
            prompt=prompt
        )
print(f"Prompt: {prompt} \nResp: {resp['response']}")

<a id="13"></a> 
### 1.3 Get response probability from LLMs

---

<!-- bilingual-zh -->

### 1.3 获取响应概率


- **Get token probability (Observe: single word could be tokenized into subwords.)**

---

<!-- bilingual-zh -->

**获取词元概率：**注意一个单词可能被切分为多个子词。


In [ ]:
model = "qwen3:0.6b" # "qwen3:1.7b"
prompt = "Fudan University is located in which city? Answer with one word."
num_top_tokens = 20 # number of top alternatives per generated token
resp = ollama.generate(
    model = model,
    prompt = prompt,
    stream = False,
    logprobs = True,
    think = False,
    top_logprobs = num_top_tokens
)
print("response:", repr(resp['response']))

# Each element usually corresponds to one generated token
for i, lp in enumerate(resp.get("logprobs", [])):
    tok = lp.get("token")
    logp = lp.get("logprob")
    p = math.exp(logp) if logp is not None else None
    print(f"{i:02d} token={tok!r:>16} logp={logp: .4f} p={p:.4f}")

- **Get specific token probability distribution via logprobs (without thinking)**

---

<!-- bilingual-zh -->

**通过 logprobs 获取具体词元的概率分布（不启用思考）。**


In [ ]:
model = "qwen3:0.6b"
prompt = "Fudan University is located in which city? Answer with one word."

res = ollama.generate(
    model=model,
    prompt=prompt,
    logprobs=True,
    think = False,
    top_logprobs=10,
    options={"temperature": 0.0, # greedy decoding, it pick the maximal token
             "num_predict": 20,
            },
)

answer = ''
lp = res["logprobs"]
tokens = [d.get("token", "") for d in lp]
print(f'We use model: {model}')
for i in range(len(lp)):
    tok = lp[i].get("token", "")
    logp = lp[i].get("logprob", None)
    alts = lp[i].get("top_logprobs", [])
    p = math.exp(logp) if logp is not None else float("nan")
    if tok == "\n" or tok == "\n\n": # stop when answer ends (often newline).
        break
    answer += tok
    print(f"--- top probabilities of token-{i:02d} ---")
    for a in alts[:20]:
        prob_a = math.exp(a['logprob'])
        print(f"{a['token']!r:>12}:{prob_a:.5f}")
    print(f"\nPartial Response: {answer}\n")
print(f"Final Response: {answer}")

- **LLMs are probabilitic distributions**

---

<!-- bilingual-zh -->

**大语言模型是概率模型。**


From the outputs above, you can see that the response from these LLMs may differ from run to run. In fact, LLMs are **probabilistic models**: given the same prompt (i.e., question), they may produce different responses (i.e., answers). We can describe this inference process using mathematical notation. Let $p_\theta(\cdot)$ denote a trained language model. Here, you can think of $p_\theta$ as **Qwen3:0.6B**, **Qwen3:1.7B**, or any other model. Given the following prompt

<blockquote style="
  background:#e8f4ff;
  border-left:4px solid #7db7ff;
  padding:10px 14px;
  border-radius:6px;
  margin: 10px 10%;">
Prompt = Fudan University is located in which city? Answer with one word.
</blockquote>

**This prompt is a sequence of tokens**. The model outputs a response, which is also a sequence of tokens. In other words, the model uses an inference procedure to predict the next token(s) conditioned on the prompt. In math, it models the conditional probability
<blockquote style="
  background:#e8f4ff;
  border-left:4px solid #7db7ff;
  padding:10px 14px;
  border-radius:6px;
  margin: 10px 10%;">$$p_\theta \left(w_{n+1} \mid w_1,w_2,\ldots,w_n\right),$$</blockquote>
where

- **Prompt**$[w_1,w_2,\ldots,w_n]$ =[ Fudan University is located in which city? Answer with one word. ],
- **Response** = $[w_{n+1}]$ (in this simplified one-word setting).

The model then outputs the most likely token $w_{n+1}$ (or samples from the distribution) using its own inference algorithm. We will discuss inference strategies in later lectures. By the definition of conditional probability, we have

<blockquote style="
  background:#e8f4ff;
  border-left:4px solid #7db7ff;
  padding:10px 14px;
  border-radius:6px;
  margin: 10px 10%;">
$$
p_\theta \left(w_{n+1} \mid w_1,w_2,\ldots,w_n\right) =
\frac{
p_\theta \left(w_1,w_2,\ldots,w_n,w_{n+1}\right)
}{
p_\theta \left(w_1,w_2,\ldots,w_n\right)
}.
$$
</blockquote>

The key takeaway is: **if we can learn a model that assigns probabilities to sequences of arbitrary length**, i.e., $p_\theta(w_1,w_2,\ldots,w_k)$ for any positive integer $k$, then conditional probabilities follow naturally from these joint probabilities. Let us do one more example to see when the response is more than one word.

---

<!-- bilingual-zh -->

上面的输出显示同一提示的响应可能因运行而异。语言模型 $p_\theta$ 根据提示词元序列 $[w_1,\ldots,w_n]$，建模下一个词元的条件概率 $p_\theta(w_{n+1}\mid w_1,\ldots,w_n)$；提示“复旦大学位于哪个城市？用一个词回答”是示例。模型可以选择最高概率词元，也可以从分布采样；解码策略将在后续课程讨论。根据条件概率定义，该概率等于包含下一词元的联合概率除以提示本身的联合概率（公式见英文原文）。

核心认识：若能为任意长度的词元序列建模联合概率，就能据此表示条件概率。接下来观察多词元响应。一个可见单词不一定恰好对应一个词元。


- **Get a sequence of token probabilities from Qwen3:1.7b**

---

<!-- bilingual-zh -->

**获取 Qwen3:1.7B 的词元概率序列。**


In [ ]:
model = "qwen3:1.7b"
prompt = "Fudan University is located in which city? Answer with one word."

res = ollama.generate(
    model=model,
    prompt=prompt,
    logprobs=True,
    think = False,
    top_logprobs=10,
    options={"temperature": 0.0, # greedy decoding, it pick the maximal token
             "num_predict": 20,
            },
)
print(res["response"])

- If we do not use thinking mode, it gives the above response. However, we can still see how the token *Shanghai* is chosen during the inference stage. You will see these subword tokens have high probabilities.

---

<!-- bilingual-zh -->

不使用思考模式时可观察上述响应及推理过程中 “Shanghai” 的子词如何被选择；检查各子词的实际概率。


In [ ]:
model = "qwen3:1.7b"
prompt = "Fudan University is located in which city? Answer with one word."
print(f'We use model: {model}')

res = ollama.generate(
    model=model,
    prompt=prompt,
    logprobs=True,
    think = False, # Do not use the thinking model.
    top_logprobs=10,
    options={"temperature": 0.0, # greedy decoding, it pick the maximal token
             "num_predict": 20,
            },
)

answer = ''
lp = res["logprobs"]
tokens = [d.get("token", "") for d in lp]
for i in range(len(lp)):
    tok = lp[i].get("token", "")
    logp = lp[i].get("logprob", None)
    alts = lp[i].get("top_logprobs", [])
    p = math.exp(logp) if logp is not None else float("nan")
    # stop when answer ends (often newline).
    if tok == "\n" or tok == "\n\n": 
        break
    answer += tok
    print(f"--- top probabilities of token-{i:02d} ---")
    for a in alts[:5]:
        prob_a = math.exp(a['logprob'])
        print(f"{a['token']!r:>5}:{prob_a:.5f}", end="")
    print(f"\nPartial Response: {answer}\n")
print(f"Final Response: {answer}")

<div style="
  background:#ffe8e8;
  border:1px solid #ff9a9a;
  border-left:6px solid #ff5c5c;
  padding:12px 14px;
  border-radius:8px;
  margin:12px 10%;
  font-weight:600;">
<center>Above response process is essentially next-token prediction task as qwen3:1.7b is an AR model! </center>
</div>

---

<!-- bilingual-zh -->

**上面的响应过程本质上是下一词元预测，因为 Qwen3:1.7B 是自回归模型。**


<a id="20"></a>
## 2. Basics for Python and spaCy

- **Python:** Use the Python version managed by the course uv environment.
- **spaCy:** Explore rule-based English and Chinese tokenization with [spaCy](https://spacy.io/).
- Compare these word boundaries with the learned subword boundaries in Section 4.

---

<!-- bilingual-zh -->

## 2. Python 与 spaCy 基础

- **Python：**使用课程 uv 环境管理的 Python 版本。
- **spaCy：**探索基于规则的英语和中文词元化。
- 将这些词边界与第 4 节学习得到的子词边界比较。


In [ ]:
# packages needed in this section
import re
import os
import spacy
import numpy as np
import matplotlib.pyplot as plt

<a id="21"></a> 
### 2.1 Unicode for text data

---

<!-- bilingual-zh -->

### 2.1 文本数据的 Unicode


- **Encoding Strings to Bytes**
  
  Use the *.encode()* method on a string object to convert it into a bytes object using UTF-8.

---

<!-- bilingual-zh -->

**字符串编码为字节：**使用字符串的 `.encode()` 将其按 UTF-8 转为 bytes 对象。


In [ ]:
my_string = "Hello, World! 你好，世界！"
encoded_bytes = my_string.encode('utf-8') # Encode the string to UTF-8 bytes
print(encoded_bytes)

- **Decoding Bytes to Strings**
  
  Use the *.decode()* method on a bytes object to convert it back into a string, specifying the encoding used to generate the bytes.

---

<!-- bilingual-zh -->

**字节解码为字符串：**使用 bytes 对象的 `.decode()`，并指定生成这些字节时使用的编码。


In [ ]:
encoded_bytes = b'Hello, World! \xe4\xbd\xa0\xe5\xa5\xbd\xef\xbc\x8c\xe4\xb8\x96\xe7\x95\x8c\xef\xbc\x81'
decoded_string = encoded_bytes.decode('utf-8') # Decode the bytes back to a string
print(decoded_string)

- **Write and Read utf-8 text into/from files**

---

<!-- bilingual-zh -->

**将 UTF-8 文本写入文件及从文件读取。**


In [ ]:
with open('unicode_file.txt', 'w', encoding='utf-8') as f:
    f.write("Some text with a special character: é, 你好，世界！")

with open('unicode_file.txt', 'r', encoding='utf-8') as f:
    content = f.read()
    print(content)

- **ASCII (American Standard Code for Information Interchange):** ASCII defines 128 character values. Each has the same numeric value in Unicode and uses one byte in UTF-8. Other Unicode characters may require multiple UTF-8 bytes.

---

<!-- bilingual-zh -->

**ASCII：**美国信息交换标准代码定义 128 个字符值；它们在 Unicode 中具有相同数值，在 UTF-8 中各占一个字节。其他 Unicode 字符可能需要多个 UTF-8 字节。


In [ ]:
import pandas as pd

rows = []
for dec in range(128):
    ch = chr(dec)
    ch_disp = repr(ch)[1:-1] if (dec < 32 or dec == 127) else ch
    rows.append({"Ch": ch_disp, "Hex": f"{dec:02X}", "Dec": dec})
df = pd.DataFrame(rows)

nblocks = 8
chunk = (len(df) + nblocks - 1) // nblocks  # ceil

blocks = []
for i in range(nblocks):
    part = df.iloc[i * chunk : (i + 1) * chunk].reset_index(drop=True)
    part.columns = pd.MultiIndex.from_product([[f"Block {i+1}"], part.columns])
    blocks.append(part)

df3 = pd.concat(blocks, axis=1)
sty = (
    df3.style
      .set_properties(**{"text-align": "center"})
      .set_table_styles([
          {"selector": "th.col_heading.level0", "props": [("text-align", "center")]},
          {"selector": "th, td", "props": [("border", "1px solid #bbb"), ("padding", "2px 6px")]},
          {"selector": "table", "props": [("border-collapse", "collapse")]},
      ])
)
sty

In [ ]:
my_string = "Hello, World! अनुच्छेद, 你好，世界！"

encodings = ["utf-8", "utf-16-le", "gb18030"] # diiferent encoding method
print("\nEncoded bytes (hex):")
for enc in encodings:
    b = my_string.encode(enc)
    print(f"{enc:9s}  {b.hex(' ')}")

# 3) Per-character UTF-8 bytes (useful for teaching)
print("\nPer-Char      Code-Point   UTF-8-Bytes")
for ch in my_string:
    b = ch.encode("utf-8")
    print(f"{repr(ch):>6}    ->    U+{ord(ch):04X}    ->    {b.hex(' ')}")

<a id="22"></a> 
### 2.2 Python basics: regular expression

---

<!-- bilingual-zh -->

### 2.2 Python 基础：正则表达式


- **Disjunction []**

---

<!-- bilingual-zh -->

**字符类 `[]`：**匹配方括号所列字符中的一个。


In [ ]:
# Task: Find woodchuck or Woodchuck : Disjunction

test_str = "This string contains Woodchuck and woodchuck."

result=re.search(pattern="[wW]oodchuck", string=test_str)
print("Matched" if result is not None else "Not found")
result=re.search(pattern=r"[wW]ooodchuck", string=test_str)
print("Matched" if result is not None else "Not found")

In [ ]:
# Find the word "woodchuck" in the following test string

test_str = "interesting links to woodchucks ! and lemurs!"
result = re.search(pattern="woodchuck", string=test_str)
print("Matched" if result is not None else "Not found")

In [ ]:
# Find !, it follows the same way:

test_str = "interesting links to woodchucks ! and lemurs!"
result = re.search(pattern="!", string=test_str)
print("Matched" if result is not None else "Not found")
result = re.search(pattern="!!", string=test_str)
print("Matched" if result is not None else "Not found")
assert re.search(pattern="!!", string=test_str) == None # match nothing

- **Disjunction any digit [0-9]**

---

<!-- bilingual-zh -->

**数字字符类 `[0-9]`：**匹配一个 ASCII 数字。


In [ ]:
# Find any single digit in a string.

result=re.search(pattern=r"[0123456789]", string="plenty of 7 to 5")
print("Matched" if result is not None else "Not found", result)
result=re.search(pattern=r"[0-9]", string="plenty of 7 to 5")
print("Matched" if result is not None else "Not found", result)

- **Negation:[^**

---

<!-- bilingual-zh -->

**否定字符类 `[^...]`：**匹配不在方括号集合中的一个字符。


In [ ]:
# Negation: If the caret ^ is the first symbol after [,
# the resulting pattern is negated. For example, the pattern 
# [^a] matches any single character (including special characters) except a.

# -- not an upper case letter
print(re.search(pattern=r"[^A-Z]", string="Oyfn pripetchik"))
# -- neither 'S' nor 's'
print(re.search(pattern=r"[^Ss]", string="I have no exquisite reason for't"))
# -- not a period
print(re.search(pattern=r"[^.]", string="our resident Djinn"))
# -- either 'e' or '^'
print(re.search(pattern=r"[e^]", string="look up ^ now"))
# -- the pattern ‘a^b’
print(re.search(pattern=r'a\^b', string=r'look up a^b now'))

- **Or Operation: |**

---

<!-- bilingual-zh -->

**或运算 `|`：**匹配多个备选模式之一。


In [ ]:
# More disjuncations with or or operation
str1 = "Woodchucks is another name for groundhog!"
result = re.search(pattern="groundhog|woodchuck",string=str1)
print(result)

In [ ]:
str1 = "Find all woodchuckk Woodchuck Groundhog groundhogxxx!"
result = re.findall(pattern="[gG]roundhog|[Ww]oodchuck",string=str1)
print(result)

- **Special Operations: ?,*,+,.,$**

---

<!-- bilingual-zh -->

**特殊运算 `?`、`*`、`+`、`.`、`$`：**分别用于可选、重复、字符匹配和位置约束等；具体语义取决于表达式上下文。


In [ ]:
# Some special chars

# ?: Optional previous char
str1 = "Find all color colour colouur colouuur colouyr"
result = re.findall(pattern="colou?r",string=str1)
print(result)

# *: 0 or more of previous char
str1 = "Find all color colour colouur colouuur colouyr"
result = re.findall(pattern="colou*r",string=str1)
print(result)

# +: 1 or more of previous char
str1 = "baa baaa baaaa baaaaa"
result = re.findall(pattern="baa+",string=str1)
print(result)
# .: any char
str1 = "begin begun begun beg3n"
result = re.findall(pattern="beg.n",string=str1)
print(result)
str1 = "The end."
result = re.findall(pattern=r"\.$",string=str1)
print(result)
str1 = "The end? The end. #t"
result = re.findall(pattern=r".$",string=str1)
print(result)

In [ ]:
# find all "the" in a raw text use above operations.

text = "If two sequences in an alignment share a common ancestor, \
mismatches can be interpreted as point mutations and gaps as indels (that \
is, insertion or deletion mutations) introduced in one or both lineages in \
the time since they diverged from one another. In sequence alignments of \
proteins, the degree of similarity between amino acids occupying a \
particular position in the sequence can be interpreted as a rough \
measure of how conserved a particular region or sequence motif is \
among lineages. The absence of substitutions, or the presence of \
only very conservative substitutions (that is, the substitution of \
amino acids whose side chains have similar biochemical properties) in \
a particular region of the sequence, suggest [3] that this region has \
structural or functional importance. Although DNA and RNA nucleotide bases \
are more similar to each other than are amino acids, the conservation of \
base pairs can indicate a similar functional or structural role."
matches = re.findall("[^a-zA-Z][tT]he[^a-zA-Z]", text)
print(matches)

In [ ]:
# Above has spaces before or after each word. A nicer way is to do the following
matches = re.findall(r"\b[tT]he\b", text)
print(matches)

- **Real-world task: match English-like words [A-Za-z]+(?:'[A-Za-z]+)?**

  You can decompose this task into:
  - [A-Za-z]+: Match one or more letters (A–Z or a–z). Examples: cat, PlayStation, Media
  - (?:...): A non-capturing group (grouping for structure, but findall won’t return it separately).
  - [A-Za-z]+: inside the group, Match a literal apostrophe ' followed by one or more letters. Examples matched as a whole word: don't, I'm, teacher's
  - ?: after the group, Makes the apostrophe+letters part optional (0 or 1 time).
  - So it matches:
    - ✅ we, Valkyria, PlayStation, don't, teacher's
    - ❌ numbers (2011), hyphenated tokens (real-time), non-ASCII letters (Senjō → only Senj), punctuation, @-@, etc.

---

<!-- bilingual-zh -->

**实际任务：匹配类似英文单词的片段 ` [A-Za-z]+(?:'[A-Za-z]+)?`**（使用时去掉代码开头的空格）。`[A-Za-z]+` 匹配一个或多个 ASCII 字母；`(?:...)` 是非捕获组，`findall` 不会把它单独作为捕获结果；组内撇号和字母可匹配 `don't`、`I'm`、`teacher's`，组后的 `?` 表示该部分可有可无。因此可匹配 `we`、`Valkyria`、`PlayStation` 和带撇号的例子；不会完整保留数字、连字符词、非 ASCII 字母、标点等，例如 `Senjō` 只匹配 `Senj`。


In [ ]:
word_re = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

text = """don't Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 ,\
lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles \
III outside Japan , is a tactical role @-@ playing video game developed by Sega and \
Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , \
it is the third game in the Valkyria series . Employing the same fusion of tactical \
and real @-@ time gameplay as its predecessors"""

tokens = word_re.findall(text)
print(len(tokens))
print(tokens)

<a id="23"></a> 
### 2.3 Tokenization using spaCy (Old way)

Usually, there are two type of tokenizations:

- **Top-down tokenization (classic)**: We define a standard and implement rules to implement that kind of tokenization.
  - word tokenization (spaCy, nltk)
  - charater tokenization (also can be done via spaCy)
    
- **Bottom-up tokenization (modern)**: We use simple statistics of letter sequences to break up words into subword tokens.
  - subword tokenization (modern LLMs use this type!)


**Tokenization via spaCy.** We assume you already installed spaCy tool. If you haven't installed yet, open your terminal and use the course uv environment, and then run the following to download English and Chinese LMs.
  
  ```uv run --extra tokenization python -m spacy download en_core_web_sm```

  ```uv run --extra tokenization python -m spacy download zh_core_web_sm```

---

<!-- bilingual-zh -->

### 2.3 用 spaCy 词元化（传统方法）

课件粗略区分两类：**自顶向下**通过定义标准和规则切词，例如 spaCy、NLTK 的词或字符切分；**自底向上**利用字符序列统计学习子词，现代大语言模型常用这类方法。这是教学上的简化分类，具体工具可能结合规则和学习方法。

原文给出安装 spaCy 英文与中文语言模型的命令；这些命令会下载模型，仅在需要并经你同意时执行。


- **Top-down (rule-based) word tokenization: Tokenization via white space**

---

<!-- bilingual-zh -->

**自顶向下的规则切词：按空格切分。**


In [ ]:
# Use split method via the whitespace " "
text = """While the Unix command sequence just removed all the numbers and punctuation"""
print(text.split(" "))

In [ ]:
# But, we have punctuations, icons, and many other small issues.

text = """Don't you love 🤗 Transformers? We sure do."""
print(text.split(" "))

- **White space cannot tokenize Chinese,Japanese,...**

---

<!-- bilingual-zh -->

**空格无法充分切分中文、日文等通常不以空格分词的文本。**


In [ ]:
text = '姚明进入总决赛'
print(text.split(" "))

- **spaCy works much better: Chinese tokenization**

---

<!-- bilingual-zh -->

**使用 spaCy 进行中文词元化。**


In [ ]:
from spacy.lang.zh import Chinese
nlp = spacy.load("zh_core_web_sm")

text = '姚明进入总决赛'
doc = nlp(text)
print([token for token in doc])

# ? can be successfully split from Transformers
text = """Don't you love 🤗 Transformers? We sure do."""
doc = nlp(text)
print([token for token in doc]) 

# Chinese Character level tokenization
nlp_ch = Chinese()
text = '姚明进入总决赛'
print([*nlp_ch(text)])

- **spaCy works much better: English tokenization**

---

<!-- bilingual-zh -->

**使用 spaCy 进行英文词元化。**


In [ ]:
import spacy
from spacy.tokens import Doc

nlp = spacy.blank("en")

text = "Hello, world!"
chars = [c for c in text]
doc = Doc(nlp.vocab, words=chars)

print([t.text for t in doc])

# ignore white space
chars = [c for c in text if not c.isspace()]
doc = Doc(nlp.vocab, words=chars)

print([t.text for t in doc])

- **spaCy handles special characters**

---

<!-- bilingual-zh -->

**spaCy 对特殊字符的处理。**


In [ ]:
text = """Special characters and numbers will need to be kept in prices ($45.55) and dates (01/02/06); 
we don’t want to segment that price into separate tokens of “45” and “55”. And there are URLs (https://www.stanford.edu),
Twitter hashtags (#nlproc), or email addresses (someone@cs.colorado.edu)."""
text = text.replace("\n", " ").strip()
nlp = spacy.load('en_core_web_sm')
doc = nlp(text)
all_tokens = [token for token in doc]
print(all_tokens)

- **Sentence-level tokenization via spaCy**

---

<!-- bilingual-zh -->

**使用 spaCy 进行句子级切分。**


In [ ]:
text = """自然语言处理（英语：Natural Language Processing，缩写作 NLP）是人工智能和语言学领域的交叉学科，\
研究计算机处理、理解与生成人类语言的技术。此领域探讨如何处理及运用自然语言；自然语言处理包括多方面和步骤，\
基本有认知、理解、生成等部分。自然语言认知和理解是让电脑把输入的语言变成结构化符号与语义关系，\
然后根据目的再处理。自然语言生成系统则是把计算机数据转化为自然语言。\
自然语言处理要研制表示语言能力和语言应用的模型, 建立计算框架来实现并完善语言模型，\
并根据语言模型设计各种实用系统及探讨这些系统的评测技术。‌‌"""
nlp = spacy.load("zh_core_web_sm")
doc = nlp(text)
print([sent.text for sent in doc.sents])

- Or, you can spaCy provided sentences function

---

<!-- bilingual-zh -->

也可以使用 spaCy 提供的句子迭代功能。


In [ ]:
 # You need to install it via: python -m spacy download zh_core_web_sm
from spacy.lang.zh.examples import sentences 
nlp = spacy.load("zh_core_web_sm")
doc = nlp(sentences[0])
text = """\
字节对编码是一种简单的数据压缩形式，这种方法用数据中不存的一个字节表示最常出现的连续字节数据。\
这样的替换需要重建全部原始数据。字节对编码实例: 假设我们要编码数据 aaabdaaabac, 字节对“aa” \
出现次数最多，所以我们用数据中没有出现的字节“Z”替换“aa”得到替换表Z <- aa 数据转变为 ZabdZabac. \
在这个数据中，字节对“Za”出现的次数最多，我们用另外一个字节“Y”来替换它（这种情况下由于所有的“Z”都将被替换，\
所以也可以用“Z”来替换“Za”），得到替换表以及数据 Z <- aa, Y <- Za, YbdYbac.
"""
doc = nlp(text)
sentences = [sent.text for sent in doc.sents]
print(sentences)

In [ ]:
# A modern and fast NLP library that includes support for sentence segmentation. 
# spaCy uses a statistical model to predict sentence boundaries, which can be more accurate 
# than rule-based approaches for complex texts.

nlp = spacy.load("en_core_web_sm")
doc = nlp("Here is a sentence. Here is another one! And the last one.")
sentences = [sent.text for sent in doc.sents]
for ind, sent in enumerate(sentences):
    print(f"sentence-{ind}: {sent}\n")

- **Lemmatization (词形还原) and Stemming (词干提取)**

---

<!-- bilingual-zh -->

**词形还原（lemmatization）与词干提取（stemming）。**


Lemmatization is the task of determining that two words have the same root, despite their surface differences. For some NLP situations, we also want two morphologically different forms of a word to behave similarly. For example in web search, someone may type the string woodchucks but a useful system might want to also return pages that mention woodchuck with no s.
  - **Example 1**: The words <span style="color:red;"><b>am</b></span>, are, and is have the shared lemma <span style="color:red;"><b>be</b></span>.
  - **Example 2**: The words <span style="color:red;"><b>dinner</b></span> and <span style="color:red;"><b>dinners</b></span> both have the lemma <span style="color:red;"><b>dinner</b></span>.

Lemmatization algorithms can be complex. For this reason we sometimes make use of a simpler but cruder method, which mainly consists of chopping off words final affixes. This naive version of morphological analysis is called stemming.

---

<!-- bilingual-zh -->

词形还原旨在将表面形式不同的词归并到相同词元形式。在某些 NLP 场景中，希望形态不同的形式表现相似：例如搜索 `woodchucks` 时也返回包含 `woodchuck` 的网页。`am`、`are`、`is` 的词元形式可为 `be`；`dinner`、`dinners` 的词元形式均为 `dinner`。词形还原算法可能复杂，因此有时使用更简单但粗糙的词干提取，主要通过截去词尾词缀进行近似形态分析。


In [ ]:
import spacy
import pandas as pd

prompt = "Fudan University is located in Shanghai. are Universities"
nlp = spacy.load("en_core_web_sm")
doc = nlp(prompt)

df = pd.DataFrame([{
    "text": t.text,
    "lemma": t.lemma_, # Lemma: The base form of the word.
    "pos": t.pos_, # POS: The simple UPOS part-of-speech tag.
    "tag": t.tag_, # Tag: The detailed part-of-speech tag.
    "dep": t.dep_, # Dep: Syntactic dependency, i.e. the relation between tokens.
    "shape": t.shape_, # Shape: The word shape – capitalization, punctuation, digits.
    "is_alpha": t.is_alpha, # is alpha: Is the token an alpha character?
    "is_stop": t.is_stop, # is stop: Is the token part of a stop list.
} for t in doc])
df

<a id="30"></a>
## 3. Datasets Exploration

---

<!-- bilingual-zh -->

## 3. 数据集探索


<a id="31"></a>
### 3.1 Heap\'s Law

---

<!-- bilingual-zh -->

### 3.1 Heaps 定律


Word types and word instances (tokens)

- **Word types** are the number of distinct words in a corpus; if the set of words in the vocabulary is
  $$
  V =\{v_1,v_2,\ldots,v_{|V|}\},
  $$
  where $|V|$ is the number of types. We also call it, the vocabulary size $|V|$. 

- **Word instances**: Given a text corpus, we treat it as a sequence of tokens $[w_1,w_2,\ldots,w_T]$ after tokenization. We call $T$, the total number $T$ of running words in this corpus.

- $T$: the number of word instances of corpus
- $|V|$: the number of word types

The larger the corpora we look at, the more word types we find, and in fact this relationship between $|V|$ and $T$ is called **Herdan's Law** or **[Heaps' Law](https://en.wikipedia.org/wiki/Heaps%27_law)** after its discoverers (in linguistics and information retrieval respectively). Given $k$ and $\beta$ positive constants, and $0<\beta<1$, it has the following form

$$
|V|=k \cdot T^\beta,
$$

the value of $\beta$ depends on the corpus size and the genre. With English text corpora, typically $k\in [10,100]$, and $\beta \in [0.4, 0.6]$. For the large corpora, $\beta$ ranges from .67 to .75. Roughly then we can say that the vocabulary size for a text goes up significantly faster than the square root of its length in words.  Let us test it!
Check more on

---

<!-- bilingual-zh -->

**词类型与词元实例：**词类型是语料中不同词的数量，即词表大小 $|V|$；文本词元序列 $[w_1,\ldots,w_T]$ 的长度 $T$ 是所有词实例的总数。随着语料增大，通常会发现更多不同词。Herdan／Heaps 定律用 $|V|=kT^\beta$ 描述经验关系，其中 $k,\beta>0$ 且 $0<\beta<1$。参数取决于语料大小及体裁；原文举出部分英语语料的示例范围（$k$ 在 10–100、$\beta$ 在 0.4–0.6；部分大型语料约 0.67–0.75）。这些是经验范围而非普遍定律。接下来进行检验。


<a id="32"></a>
### 3.2 Explore wikitext-2 dataset

---

<!-- bilingual-zh -->

### 3.2 探索 WikiText-2 数据集


**Explore WikiText-2**

The next cell downloads the dataset from Hugging Face. If the official endpoint is unavailable on your network, configure an institution-approved mirror through `HF_ENDPOINT` before starting Python. Cached datasets can be reused.

---

<!-- bilingual-zh -->

**探索 WikiText-2：**下一单元会从 Hugging Face 下载数据集。若官方端点在你的网络不可用，可在启动 Python 前通过 `HF_ENDPOINT` 配置机构允许的镜像；已有缓存可复用。


In [ ]:
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

from datasets import load_dataset
# loads train/validation/test
wiki2_ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
wiki2_train = wiki2_ds["train"]

In [ ]:
summary = pd.DataFrame([
    {
        "split": name,
        "num_rows": d.num_rows,
        "num_columns": d.num_columns,
        "columns": ", ".join(d.column_names),
    }
    for name, d in wiki2_ds.items()
])
summary

In [ ]:
preview = []
for name, d in wiki2_ds.items():
    for i in range(5):
        preview.append({"split": name, "idx": i, **d[i]})

pd.DataFrame(preview)

- **Simple tokenization and testing the Heap's law.**

---

<!-- bilingual-zh -->

**简单切分并检验 Heaps 定律。**


In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

start_time = time.time()
wiki2_train = wiki2_ds["train"]
word_re = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?") # simple word tokenizer

def heaps_curve(dataset, step=1000):
    V = set()
    N = 0
    Ns, Vs = [], []
    for ex in dataset:
        text = ex["text"]
        if not text or text.strip() == "": # empty word -> continue
            continue
        words = word_re.findall(text.lower()) # make all words to low cases
        for w in words:
            N += 1
            V.add(w)
            if N % step == 0:
                Ns.append(N)
                Vs.append(len(V))
    return np.array(Ns), np.array(Vs)

Ns, Vs = heaps_curve(wiki2_train, step=1000)

# Fit log |V| = log k + beta log N  -> linear regression on logs
logN = np.log(Ns)
logV = np.log(Vs)
beta, logk = np.polyfit(logN, logV, 1)
k = np.exp(logk)

print(f"Fitted Heaps' law: |V| ≈ {k:.2f} * T^{beta:.3f}")
print(f"Total runtime: {time.time() - start_time:.3f} seconds")

# Plot (log-log)
plt.figure(figsize=(7, 5))
plt.loglog(Ns, Vs, marker='o', linestyle='none', markersize=5, label="Empirical (Wikitext-2)")

# fitted line
Ns_line = np.linspace(Ns[0], Ns[-1], 200)
Vs_line = k * (Ns_line ** beta)
plt.loglog(Ns_line, Vs_line, label=fr"Fitted ($k={k:.3f},\beta=${beta:.3f})")

plt.xlabel("$T$ (tokens)", fontsize = 15)
plt.ylabel("$|V|$ (vocab)", fontsize = 15)
plt.title(r"Heaps' Law (Simple tokenization): $|V|=k \cdot T^{\beta}$", fontsize = 15)
plt.legend(fontsize = 15)
plt.tight_layout()
plt.show()

- **Use spaCy tokenization and test Heap's Law**

---

<!-- bilingual-zh -->

**使用 spaCy 切分并检验 Heaps 定律。**


In [ ]:
start_time = time.time()

nlp = spacy.load(
    "en_core_web_sm", 
    disable=["tagger","parser","ner","lemmatizer"]
)
wiki2_train = wiki2_ds["train"]

def heaps_curve_spacy(dataset, step=10_000, batch_size=256):
    V = set()
    N = 0
    Ns, Vs = [], []

    texts = (ex["text"] for ex in dataset if ex["text"] and ex["text"].strip())
    for doc in nlp.pipe(texts, batch_size=batch_size):
        for tok in doc:
            if tok.is_alpha: # choose your definition of "word"
                w = tok.text.lower()
                N += 1
                V.add(w)
                if N % step == 0:
                    Ns.append(N)
                    Vs.append(len(V))
    return np.array(Ns), np.array(Vs)

Ns, Vs = heaps_curve_spacy(wiki2_train, step=1000)

# Fit log |V| = log k + beta log N
logN = np.log(Ns)
logV = np.log(Vs)
beta, logk = np.polyfit(logN, logV, 1)
k = np.exp(logk)

print(f"Fitted Heaps' law (spaCy): |V| ≈ {k:.2f} * T^{beta:.3f}")
print(f"Total runtime: {time.time() - start_time:.3f} seconds")
# Plot (log-log)
plt.figure(figsize=(7, 5))
plt.loglog(Ns, Vs, marker='o', linestyle='none', markersize=5, label="Empirical (Wikitext-2)")

# fitted line
Ns_line = np.linspace(Ns[0], Ns[-1], 200)
Vs_line = k * (Ns_line ** beta)
plt.loglog(Ns_line, Vs_line, label=fr"Fitted ($k={k:.3f},\beta=${beta:.3f})")

plt.xlabel("$T$ (tokens)", fontsize = 15)
plt.ylabel("$|V|$ (vocab)", fontsize = 15)
plt.title(r"Heaps' Law (Spacy tokenization): $|V|=k\cdot T^{\beta}$", fontsize = 15)
plt.legend(fontsize = 15)
plt.tight_layout()
plt.show()

<a id="33"></a>
### 3.3 Explore wikitext-103 dataset

---

<!-- bilingual-zh -->

### 3.3 探索 WikiText-103 数据集


The next cell loads the larger WikiText-103 dataset using the same `datasets` interface. This requires a dataset download unless it is cached. Prepare it before running tokenizer training in Section 4; classroom E01–E06 exercises do not require it.

---

<!-- bilingual-zh -->

下一单元通过相同的 `datasets` 接口加载更大的 WikiText-103；若没有缓存需要下载。第 4 节训练词元化器前需准备它，课堂 E01–E06 不需要。


- **Explore wikitext-103**

---

<!-- bilingual-zh -->

**探索 WikiText-103。**


In [ ]:
from datasets import load_dataset
wiki103_ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")  # will hit cache if present

In [ ]:
summary = pd.DataFrame([
    {
        "split": name,
        "num_rows": d.num_rows,
        "num_columns": d.num_columns,
        "columns": ", ".join(d.column_names),
    }
    for name, d in wiki103_ds.items()
])
summary

In [ ]:
start_time = time.time()
wiki103_train = wiki103_ds["train"]
word_re = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

def heaps_curve(dataset, step=1000):
    V = set()
    N = 0
    Ns, Vs = [], []
    for ex in dataset:
        text = ex["text"]
        if not text or text.strip() == "": # empty word -> continue
            continue
        words = word_re.findall(text.lower()) # make all words to low cases
        for w in words:
            N += 1
            V.add(w)
            if N % step == 0:
                Ns.append(N)
                Vs.append(len(V))
    return np.array(Ns), np.array(Vs)

Ns, Vs = heaps_curve(wiki103_train, step=1000)

# Fit log |V| = log k + beta log N  -> linear regression on logs
logN = np.log(Ns)
logV = np.log(Vs)
beta, logk = np.polyfit(logN, logV, 1)
k = np.exp(logk)

print(f"Fitted Heaps' law (spaCy): |V| ≈ {k:.2f} * T^{beta:.3f}")
print(f"Total runtime: {time.time() - start_time:.3f} seconds")
# Plot (log-log)
plt.figure(figsize=(7, 5))
plt.loglog(Ns, Vs, marker='o', linestyle='none', markersize=5, label="Empirical (Wikitext-103)")

# fitted line
Ns_line = np.linspace(Ns[0], Ns[-1], 200)
Vs_line = k * (Ns_line ** beta)
plt.loglog(Ns_line, Vs_line, label=fr"Fitted ($k={k:.3f},\beta=${beta:.3f})")

plt.xlabel("$N$ (tokens)", fontsize = 15)
plt.ylabel("$|V|$ (vocab)", fontsize = 15)
plt.title(r"Heaps' Law (Spacy tokenization): $|V|=k \cdot T^{\beta}$", fontsize = 15)
plt.legend(fontsize = 15)
plt.tight_layout()
plt.show()

<a id="34"></a>
### 3.4 BookCorpus datasets (GPT-1)

---

<!-- bilingual-zh -->

### 3.4 BookCorpus 数据集（GPT-1）


BookCorpus was used in training GPT-1; see the [GPT-1 paper](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf).

**Optional archive exploration:** use a locally available archive and set `BOOKCORPUS_ARCHIVE` to its path. Keep datasets in `workspace/`. The following statistics example uses the prepared measurements from the original course notebook and does not require the full archive.

---

<!-- bilingual-zh -->

BookCorpus 曾用于 GPT-1 训练，原文链接其论文。**可选归档探索：**使用本地已有归档，将路径设为 `BOOKCORPUS_ARCHIVE`，数据集保存在 `workspace/`。下面统计示例使用原课程 notebook 预先测得的数据，不需要完整归档。


In [ ]:
import time
import tarfile
from pathlib import Path

path = Path(os.environ.get("BOOKCORPUS_ARCHIVE", "../.datasets/books1.tar.gz"))
start_time = time.time()
if path.is_file():
    with tarfile.open(path, "r:gz") as tar:
        names = tar.getnames()
        print("num members:", len(names))
        print("\n".join(names[:20]))
    print(f"total time to scan BookCorpus: {time.time() - start_time:.2f} seconds")
else:
    print("Optional: set BOOKCORPUS_ARCHIVE to a local archive to inspect its members.")

The original course's [tokenization_bookcorpus.py](https://github.com/baojian/llm-26/blob/main/lecture-01-tokenization/tokenization_bookcorpus.py) generated the prepared Heaps' law statistics used below. The next cell downloads that small statistics file into your personal `assets/` folder when it is missing; it does not download the corpus.

---

<!-- bilingual-zh -->

原课程的 `tokenization_bookcorpus.py` 生成下方预备的 Heaps 定律统计。个人 `assets/` 文件夹缺少该小统计文件时，下一单元会下载它；不会下载整个语料。


- **Heap's Law of BookCorpus dataset**

---

<!-- bilingual-zh -->

**BookCorpus 数据集的 Heaps 定律。**


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

def load_heaps_jsonl(log_path: str):
    Ns, Vs = [], []
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            Ns.append(r["total_tokens"])
            Vs.append(r["vocab_size"])
    return np.array(Ns), np.array(Vs)

from pathlib import Path
from urllib.request import urlopen

statistics_path = Path("assets/books1_wordcounts.heaps_log.jsonl")
if not statistics_path.exists():
    statistics_url = (
        "https://raw.githubusercontent.com/baojian/llm-26/main/"
        "lecture-01-tokenization/assets/books1_wordcounts.heaps_log.jsonl"
    )
    with urlopen(statistics_url, timeout=30) as response:
        statistics_path.parent.mkdir(parents=True, exist_ok=True)
        statistics_path.write_bytes(response.read())
Ns, Vs = load_heaps_jsonl(statistics_path)

# fit log |V| = log k + beta log N
beta, logk = np.polyfit(np.log(Ns), np.log(Vs), 1)
k = np.exp(logk)
print(f"Fit: |V| ≈ {k:.2f} * T^{beta:.3f}")


plt.figure(figsize=(7,5))

plt.loglog(Ns, Vs, marker="o", linestyle="none", 
           markersize=5, label="Empirical (BookCorpus)")
Ns_line = np.linspace(Ns[0], Ns[-1], 200)
plt.loglog(Ns_line, k * (Ns_line ** beta), label=fr"Fitted Curve $(k={k:.3f},\beta=${beta:.3f})")
plt.xlabel("$T$ (tokens)", fontsize = 14)
plt.ylabel("$|V|$ (vocab)", fontsize = 14)
plt.title(r"Heaps' Law (Simple tokenization): $|V|=k \cdot T^{\beta}$", fontsize = 14)
plt.legend(fontsize = 14)
plt.tight_layout()
plt.show()

<a id="40"></a>
## 4. LLMs tokenization (Modern Way)

In general, there are two type of tokenizations

- **Top-down tokenization**: It defines a standard and implement rules to implement that kind of tokenization. It includes, for example, word tokenization, charater tokenization, and others.
  
- **Bottom-up tokenization**: It uses simple statistics of letter sequences to break up words into subword tokens. Subword tokenization (modern LLMs use this type!) belongs to this type.

In case of bottom-up tokenization, there are three standard way to do modern LLM tokenization:

- **BPE tokenization:** <a href="/files/papers/BPE-A_New_Algorithm_for_Data_Compression-1994.pdf"
   target="_blank" rel="noopener">[Original BPE paper]</a>, <a href="/files/papers/GPT2-Language_Models_are_Unsupervised_Multitask_Learners-2019.pdf"
   target="_blank" rel="noopener">[Adopted in GPT-2]</a>
  
- **Wordpiece tokenization:** <a href="/files/papers/Wordpiece-Japanese_and_Korean_voice_search-1994.pdf"
   target="_blank" rel="noopener">[Original Wordpiece paper]</a>, <a href="/files/papers/GNMT-Google_Neural_Machine_Translation_System_Bridging_the_Gap_between_Human_and_Machine_Translation-2016.pdf"
   target="_blank" rel="noopener">[Adopted in GNMT]</a>, <a href="/files/papers/BERT-Pre-training_of_Deep_Bidirectional_Transformers_for_Language_Understanding-2018.pdf"
   target="_blank" rel="noopener">[Adopted in BERT]</a>

- **Unigram tokenization:** <a href="/files/papers/Unigram-Subword_Regularization_Improving_Neural_Network_Translation_Models_with_Multiple_Subword_Candidates-2018.pdf"
   target="_blank" rel="noopener">[Original Unigram paper]</a>, <a href="/files/papers/ALBERT-A_Lite_BERT_for_Self-supervised_Learning_of_Language_Representations-2020.pdf"
   target="_blank" rel="noopener">[Adopted in AlBERT]</a>, <a href="/files/papers/T5-model-Exploring_the_Limits_of_Transfer_Learning_with_a_Unified_Text-to-Text_Transformer-2020.pdf"
   target="_blank" rel="noopener">[Adopted in T5]</a>

---

<!-- bilingual-zh -->

## 4. 大语言模型词元化（现代方法）

原文以教学方式区分自顶向下的规则切分（如词或字符切分）与自底向上的统计子词方法。常见子词方法包括 **BPE**（学习相邻片段合并）、**WordPiece**（使用词表片段，常见 BERT 编码采用最长匹配）、**Unigram**（通过片段概率比较可能切分）。原文链接相关论文和应用；不同工具可能结合预处理、规则及训练方法。


In [ ]:
import tiktoken

<a id="41"></a>
### 4.1 BPE tokenization

---

<!-- bilingual-zh -->

### 4.1 BPE 词元化


- **Explore trained tokenizer: [tiktoken](https://github.com/openai/tiktoken)** Any trained tokenizer has two functions:
  - **encode**: it maps running text into token ids (integers)
  - **decode**: it maps token id back to each text token (strings)

- It has been adopted from all modern LLMs including ChatGPT, GPT-series, and many others.
- tiktoken uses a byte-level BPE. (The emoji like 🤗 could be encoded as multiple UTF-8 bytes.)

---

<!-- bilingual-zh -->

**探索已训练的词元化器 tiktoken：**编码（encode）将文本映射为词元整数 ID；解码（decode）将 ID 序列还原为文本。tiktoken 使用字节级 BPE；例如表情 🤗 可能涉及多个 UTF-8 字节。不同模型使用的具体编码应以模型及工具文档为准。


#### 4.1.1 explore tiktoken tokenizer

The [tiktoken tokenizer](https://github.com/openai/tiktoken).

---

<!-- bilingual-zh -->

#### 4.1.1 探索 tiktoken 词元化器


In [ ]:
# Load cl100k_tokenizer (it was used by gpt-3.5-turbo family, gpt-4, and also gpt-4-turbo

cl100k_tokenizer = tiktoken.get_encoding("cl100k_base")

text = "Don't you love 🤗 Transformers? We sure do."
token_ids = cl100k_tokenizer.encode(text)
token_strs = [cl100k_tokenizer.decode([tid]) for tid in token_ids]

print("ids:", token_ids)
print("tokens:", token_strs)

- what you’re seeing (' �', '�', '�') is normal. Those are individual bytes / partial UTF-8 pieces that don’t form a valid standalone Unicode character when decoded one token at a time.
- Next, we explore o200k_base, which is the tokenizer (encoding) used by OpenAI’s newer “o-series” models, especially the GPT-4o family. If you call a model like gpt-4o / gpt-4o-mini, the tokenization is o200k_base.

---

<!-- bilingual-zh -->

看到单独解码出的 `�` 是正常现象：单个词元可能只包含 UTF-8 字节的一部分，不能独立构成有效 Unicode 字符。接下来探索 `o200k_base` 编码；原文将其与 GPT-4o 家族关联。具体模型与版本的编码应以官方映射为准。


In [ ]:
gpt35_tokenizer = tiktoken.get_encoding("o200k_base")

token_ids = gpt35_tokenizer.encode(text)
token_strs = [gpt35_tokenizer.decode([tid]) for tid in token_ids]

print("ids:", token_ids)
print("tokens:", token_strs)

# Count tokens by counting the length of the list returned by .encode().
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens
print("---")

text = "Don't you love 🤗 Transformers? We sure do."
print(f'\"{text}\" has been encoded into {num_tokens_from_string(text, "cl100k_base")} subwords')
print(f'\"{text}\" has been encoded into {num_tokens_from_string(text, "o200k_base")} subwords')

- Note the tokens generated above is slightly different from ones generated by cl100k_base.

---

<!-- bilingual-zh -->

注意上方词元与 `cl100k_base` 的结果略有不同。


In [ ]:
# .decode() converts a list of token integers to a string.
encode_ids = [83, 1609, 5963, 374, 2294, 0, 11, 2025]
print(f'the decoded string is: \"{gpt35_tokenizer.decode(encode_ids)}\"')

- Try a slightly long text.

---

<!-- bilingual-zh -->

尝试一段稍长的文本。


In [ ]:
text = """
Chapters 5 to 8 teach the basics of 🤗 Datasets and 🤗 Tokenizers before diving into classic NLP tasks.\
By the end of this part, you will be able to tackle the most common NLP problems by yourself. \
By the end of this part, you will be ready to apply 🤗 Transformers to (almost) any machine \
learning problem! E=mc^2. f(x) = x^2+y^2, print('hello world!’) baojianzhou. asdasfasdgasdg
"""
token_ids = gpt35_tokenizer.encode(text)
token_strs = [gpt35_tokenizer.decode([tid]) for tid in token_ids]
print(token_ids)
print(token_strs)

- **Limitations of general tokenizers**

**Generic tokenizers can split domain strings (genes/SMILES) in semantically harmful ways; we often need domain-specific tokenization (k-mers / specialized vocabularies) or character-level modeling.** Let us consider the following example

---

<!-- bilingual-zh -->

**通用词元化器的局限：**通用词元化器可能将基因名或 SMILES 等领域字符串切分成不利于语义建模的片段；可考虑领域专用词表、k-mer 或字符级建模，并通过任务实验评估。


In [ ]:
text = """acetylsalicylic acid, aspirin, Group of substances: organic Physical appearance: \
colorless needles crystals Empirical formula (Hill's system for organic substances): \
C9H8O4 Structural formula as text: CH3COOC6H4COOH, 
the smiles format of it is CC(=O)OC1=CC=CC=C1C(O)=O."""

token_ids = gpt35_tokenizer.encode(text)
token_strs = [gpt35_tokenizer.decode([tid]) for tid in token_ids]
print(token_ids)
print(token_strs) # CC(=O)OC1=CC=CC=C1C(O)=O. should be a whole, but it is tokenized into small pieces.

**You see that CC(=O)OC1=CC=CC=C1C(O)=O is better to be a single token, but the standard tokenization has been split it into serval parts.**

---

<!-- bilingual-zh -->

原文以 `CC(=O)OC1=CC=CC=C1C(O)=O` 为例，观察标准词元化器将其分成多个部分。是否应将整个分子表达式作为单一词元取决于任务、词表与实验证据，不能仅凭切分结果断定。


#### 4.1.2 train a BPE tokenizer using tokenizers

---

<!-- bilingual-zh -->

#### 4.1.2 使用 tokenizers 训练 BPE 词元化器


**Training BPE with Hugging Face Tokenizers**

Use the WikiText-103 **training split** to learn the vocabulary; reserve validation and test text for evaluation.

- Start with characters observed in the training corpus.
- Count adjacent token pairs and merge frequent pairs.
- Stop at the requested vocabulary size or when no merge remains.

Runtime depends on corpus size and hardware. This character-initialized example differs from the byte-initialized classroom tokenizer.

---

<!-- bilingual-zh -->

**使用 Hugging Face Tokenizers 训练 BPE：**仅用 WikiText-103 的**训练集**学习词表，验证和测试文本留作评估。先从训练语料观察到的字符开始，统计相邻词元对并合并高频对，达到目标词表大小或没有可合并项时停止。耗时取决于语料与硬件。本例以字符初始化，与课堂上以字节初始化的词元化器不同。


- **Step 1: pretraining for tokenization learner (adding whitespace)**

---

<!-- bilingual-zh -->

**步骤 1：配置词元化学习器的预处理（空白切分）。**


In [ ]:
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# load wikitext-103
wiki103_ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")  # train/validation/test

# Step 1: pre-training
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

- **Step 2: training (using BPE algorithm)**

---

<!-- bilingual-zh -->

**步骤 2：使用 BPE 算法训练。**


In [ ]:
start_time = time.time()

# build tokenizer + trainer
trainer = BpeTrainer(
    vocab_size=30000,
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]", "[MASK]", "[CLS]", "[SEP]"]
)
# Train only on the training split; keep held-out text for evaluation.
def batch_iterator(batch_size=1000):
    for split in ["train"]:
        for i in range(0, len(wiki103_ds[split]), batch_size):
            yield wiki103_ds[split][i:i+batch_size]["text"]
tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)
# It will take above 30 seconds
print(f"Total runtime of training tokenizer on wikitext-103: {time.time() - start_time:.3f} seconds")

- **Step 3: save the tokenizer**

---

<!-- bilingual-zh -->

**步骤 3：保存词元化器。**


In [ ]:
# Step 3: save the trained tokenizer
tokenizer.save("assets/wikitext103-bpe.json")

In [ ]:
tokenizer = Tokenizer.from_file("assets/wikitext103-bpe.json")
output = tokenizer.encode("Hello, y'all! How are you 😁 ?")
print(output.tokens)
print(output.ids)

<a id="42"></a>
### 4.2 WordPiece tokenization

---

<!-- bilingual-zh -->

### 4.2 WordPiece 词元化


- **Step 1: normalization**

---

<!-- bilingual-zh -->

**步骤 1：规范化。**


In [ ]:
# Step 1: normalization, Normalization is, in a nutshell, a set of operations you apply to 
# a raw string to make it less random or “cleaner”. Common operations include stripping 
# whitespace, removing accented characters or lowercasing all text. 

from tokenizers import normalizers
from tokenizers.normalizers import NFD, StripAccents
normalizer = normalizers.Sequence([NFD(), StripAccents()])

print(normalizer.normalize_str("Héllò hôw are ü?"))
tokenizer.normalizer = normalizer

- **Step 2: Pre-Tokenization**

---

<!-- bilingual-zh -->

**步骤 2：预词元化。**


In [ ]:
# Step 2: Pre-Tokenization, Pre-tokenization is the act of splitting a text into smaller 
# objects that give an upper bound to what your tokens will be at the end of training. 
# A good way to think of this is that the pre-tokenizer will split your text into “words”
# and then, your final tokens will be parts of those words. An easy way to pre-tokenize 
# inputs is to split on spaces and punctuations, which is done by the pre_tokenizers.
from tokenizers.pre_tokenizers import Whitespace
pre_tokenizer = Whitespace()
pre_tokenizer.pre_tokenize_str("Hello! How are you? I'm fine, thank you.")

In [ ]:
from tokenizers import pre_tokenizers
from tokenizers.pre_tokenizers import Digits
pre_tokenizer = pre_tokenizers.Sequence([Whitespace(), Digits(individual_digits=True)])
pre_tokenizer.pre_tokenize_str("Call 911!")

- **Step 3: Choose your model (WordPiece)**

---

<!-- bilingual-zh -->

**步骤 3：选择模型（WordPiece）。**


Choose your tokenization algorithm. 

- models.BPE
- models.Unigram
- models.WordLevel
- models.WordPiece

---

<!-- bilingual-zh -->

选择词元化算法：`models.BPE`、`models.Unigram`、`models.WordLevel` 或 `models.WordPiece`。


- **Step 4: Post-Processing**

Post-processing is the last step of the tokenization pipeline, to perform any additional transformation to the Encoding before it’s returned, like adding potential special tokens.

---

<!-- bilingual-zh -->

**步骤 4：后处理。**这是词元化流水线的最后一步，在返回编码前执行额外转换，例如添加可能需要的特殊词元。


In [ ]:
from tokenizers.processors import TemplateProcessing
tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", 1), ("[SEP]", 2)],
)

In [ ]:
# BERT tokenization building
import time
start_time = time.time()

from transformers import AutoTokenizer
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
bert_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))

from tokenizers import normalizers
from tokenizers.normalizers import NFD, Lowercase, StripAccents
bert_tokenizer.normalizer = normalizers.Sequence([NFD(), Lowercase(), StripAccents()])

from tokenizers.pre_tokenizers import Whitespace
bert_tokenizer.pre_tokenizer = Whitespace()

from tokenizers.processors import TemplateProcessing
bert_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[
        ("[CLS]", 1),
        ("[SEP]", 2),
    ],
)
from tokenizers.trainers import WordPieceTrainer
trainer = WordPieceTrainer(vocab_size=30522, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

# Train only on the training split; keep held-out text for evaluation.
def batch_iterator(batch_size=1000):
    for split in ["train"]:
        for i in range(0, len(wiki103_ds[split]), batch_size):
            yield wiki103_ds[split][i:i+batch_size]["text"]
bert_tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)
# It will take above 1 second
print(f"Total runtime of training tokenizer on wikitext-103: {time.time() - start_time:.3f} seconds")
bert_tokenizer.save("assets/wikitext103-bert.json")

- **Step 5: Encoding and Decoding**

---

<!-- bilingual-zh -->

**步骤 5：编码与解码。**


In [ ]:
output = bert_tokenizer.encode("Hello, y'all! How are you 😁 ?")
print(output.ids)
# [1, 27462, 16, 67, 11, 7323, 5, 7510, 7268, 7989, 0, 35, 2]
bert_tokenizer.decode([1, 27462, 16, 67, 11, 7323, 5, 7510, 7268, 7989, 0, 35, 2])
# "Hello , y ' all ! How are you ?"

In [ ]:
output = bert_tokenizer.encode("Welcome to the 🤗 Tokenizers library.")
print(output.tokens)
# ["[CLS]", "welcome", "to", "the", "[UNK]", "tok", "##eni", "##zer", "##s", "library", ".", "[SEP]"]
bert_tokenizer.decode(output.ids)
# "welcome to the tok ##eni ##zer ##s library ."

<a id="43"></a>
### 4.3 Unigram tokenization

---

<!-- bilingual-zh -->

### 4.3 Unigram 词元化


<a id="44"></a>
### 4.4 Huggingface pretrained tokenizers

---

<!-- bilingual-zh -->

### 4.4 Hugging Face 预训练词元化器


PreTrainedTokenizer, PreTrainedTokenizerBase, AutoTokenizer

- https://github.com/huggingface/transformers/blob/main/src/transformers/tokenization_utils.py
- https://github.com/huggingface/transformers/blob/main/src/transformers/tokenization_utils_base.py
- https://github.com/huggingface/transformers/blob/main/src/transformers/tokenization_utils_fast.py
- Check tokenizers at https://github.com/huggingface/transformers/blob/main/setup.py
- If you want to train a tokenizer by yourself, then go to: https://github.com/huggingface/tokenizers
- A fast BPE tokenizer is also at: https://github.com/openai/tiktoken
- An implementation of sentencepiece is at: https://github.com/google/sentencepiece
- There are 3 most common methods for tokenization: https://github.com/huggingface/tokenizers/tree/main/tokenizers/src/models
- - BPE: https://aclanthology.org/P16-1162.pdf
  - Unigram: https://arxiv.org/pdf/1804.10959
  - WordPiece https://static.googleusercontent.com/media/research.google.com/en//pubs/archive/37842.pdf

---

<!-- bilingual-zh -->

**预训练词元化器参考：**原文提供 `PreTrainedTokenizer`、`PreTrainedTokenizerBase`、`AutoTokenizer` 及快速实现的源码链接，也列出 Hugging Face Tokenizers、OpenAI tiktoken、SentencePiece 和 BPE／Unigram／WordPiece 的论文。加载远程预训练词元化器可能需要网络和下载词表；使用前检查代码及缓存情况。


In [ ]:
from transformers import Qwen2Tokenizer, Qwen2TokenizerFast

In [ ]:
tokenizer = Qwen2TokenizerFast.from_pretrained(pretrained_model_name_or_path="Qwen/Qwen-tokenizer")
text = "What is natural language processing? 🤗"
encoded = tokenizer(text, return_tensors="pt")
print("Input IDs:", encoded["input_ids"])

In [ ]:
from transformers import AutoTokenizer
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-V3")
# Example: tokenize some text
text = "What is natural language processing? 🤗"
encoded = tokenizer(text, return_tensors="pt")
print("Input IDs:", encoded["input_ids"])
decoded = tokenizer.decode(encoded["input_ids"][0])
print("Decoded text:", decoded)
decoded = tokenizer.decode(encoded["input_ids"][0], skip_special_tokens=True)
print("Decoded with special text:", decoded)

In [ ]:
vocab = tokenizer.get_vocab()
print("Vocab size (attribute):", tokenizer.vocab_size)
print("Vocab size (dict length):", len(vocab))
id_to_token = {id_: tok for tok, id_ in vocab.items()}
pairs = [[i, repr(id_to_token[i])]for i in range(20)] # show first 20 tokens
print(pairs)

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-72B-Instruct")
text = "What is natural language processing? 🤗"
encoded = tokenizer(text, return_tensors="pt")
print("Input IDs:", encoded["input_ids"])
# Decode including special tokens
decoded = tokenizer.decode(encoded["input_ids"][0])
print("Decoded text:", decoded)

In [ ]:
from transformers import AutoTokenizer
# Download vocabulary from huggingface.co and cache.
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
# Download vocabulary from huggingface.co (user-uploaded) and cache.
tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-german-cased")
# If vocabulary files are in a directory (e.g. tokenizer was saved using *save_pretrained('./test/saved_model/')*)
# tokenizer = AutoTokenizer.from_pretrained("./test/bert_saved_model/")
# Download vocabulary from huggingface.co and define model-specific arguments
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base", add_prefix_space=True)

<a id="50"></a>
## 5. Save your work

---

<!-- bilingual-zh -->

## 5. 保存你的工作


In [ ]:
# Record the time spent exploring this notebook.
jnk_end_time = time.time()
total = jnk_end_time - jnk_start_time
h, rem = divmod(int(total), 3600)
m, s = divmod(rem, 60)
print(f"Total runtime: {h} h {m} min {s} s")

Save your notebook in JupyterLab. These are ungraded practice exercises; follow the [Fall course assessment instructions](https://baojian.github.io/llm-26-fall/#assessment) for any graded submissions. To export a saved copy, run the next cell.

---

<!-- bilingual-zh -->

在 JupyterLab 保存 notebook。这些是不计分练习；计分提交请遵循原文链接的秋季课程考核说明。要导出已保存副本，运行下一个单元。


In [ ]:
!jupyter nbconvert lecture-01-exercise-tokenization.ipynb --to html --template classic --embed-images

<a id="60"></a>
## 6. Useful commands

---

<!-- bilingual-zh -->

## 6. 常用命令


Export the saved notebook from its personal working directory:

```sh
jupyter nbconvert lecture-01-exercise-tokenization.ipynb --to html --template classic --embed-images
```

Open the exported HTML in a browser to print it as a PDF. From the repository root, refresh the optional environment with `uv sync --extra tokenization`. The dependency versions are recorded in `pyproject.toml` and `uv.lock`.

---

<!-- bilingual-zh -->

在个人工作目录运行上方 `jupyter nbconvert` 命令，可导出带嵌入图片的 HTML；用浏览器打开后可打印为 PDF。要刷新可选环境，在仓库根目录运行 `uv sync --extra tokenization`。依赖版本记录在 `pyproject.toml` 与 `uv.lock`。
